# Plateau’s Problem for Enneper surface

This notebook provides an implementation of the Plateau's problem, which finds a minimal surface shape that connects a set of interfaces.
More details on this example, can be found in [our paper](https://arxiv.org/abs/2402.14009), Sections 4.1 and A.2.

### Imports and setup

In [12]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from tqdm.notebook import trange
import k3d
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from external.preconditioning.model_architecture import GeneralNet
from external.preconditioning.optimizers import HandNGD 

## Point-sampler
from GINN.numerical_boundary import find_boundary_points_numerically_with_binsearch
from models.net_w_partials import NetWithPartials
from util.sample_utils import precompute_sample_grid

torch.manual_seed(3)

device = 'cuda' #if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

In [13]:
from util.visualization.utils_mesh import get_mesh

# Parametric equations for Enneper's minimal surface in polar coordinates
def enneper_surface_polar(r, phi):
    x = r * torch.cos(phi) - (1/3) * r**3 * torch.cos(3 * phi)
    y = r * torch.sin(phi) + (1/3) * r**3 * torch.sin(3 * phi)
    z = r**2 * torch.cos(2 * phi)
    return x, y, z

    
def enneper_level_set(x):
    x1 = x[:, 0]
    y1 = x[:, 1]
    z1 = x[:, 2]
    
    term1 = (y1**2 - x1**2) / (2 * z1) + (2 / 9) * z1**2 + 2 / 3
    term2 = (y1**2 - x1**2) / (4 * z1) - (1 / 4) * (x1**2 + y1**2 + (8 / 9) * z1**2) + 2 / 9
    
    return term1**3 - 6 * term2**2


# Bounds and number of samples
n = 1000
r_max = 1.0
phi = torch.linspace(-torch.pi, torch.pi, n, dtype=torch.float64)

# Generate boundary points with r constant and phi ranging from -pi to pi
r_constant = torch.full_like(phi, r_max, dtype=torch.float64)
x, y, z = enneper_surface_polar(r_constant, phi)
pts_boundary = torch.vstack([x, y, z]).T

# Generate the mesh using the level set function
verts, faces = get_mesh(
    enneper_level_set, N=128, device=device,
    bbox_min=torch.tensor([-3, -3, -3], dtype=torch.float64),
    bbox_max=torch.tensor([3, 3, 3], dtype=torch.float64),
    chunks=2
)

# Visualization
color = 0xbbbbbb

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=color, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)

fig.display()


Output()

### Define the boundary rectangles

In [14]:
# pts_space = torch.rand([1000, 3], dtype=torch.float64)
# pts_space_fixed = torch.rand([1000, 3], dtype=torch.float64)
pts_space = 2 * torch.rand([1000, 3], dtype=torch.float64) - 1
pts_space_fixed = 2 * torch.rand([1000, 3], dtype=torch.float64) - 1
pts_eikonal = torch.cat((pts_space_fixed, pts_boundary))
pts_surface = pts_boundary
# pts_eikonal = pts_boundary


# loss_weights = {"interface": 10.0, "eikonal": 0.001, "laplacian": 1.0, "curvature": 0.01}

loss_weights = {"interface": 10.0, "eikonal": 0.001, "laplacian": 1.0, "curvature": 0.0}
# loss_weights = {"interface": 10.0, "eikonal": 0.1, "curvature": 1.0}



config = {
    "pts_boundary": pts_boundary,
    "pts_eikonal": pts_eikonal,
    "pts_space": pts_space,
    "pts_surface": pts_surface,
    "loss_weights": loss_weights,
    "regularization": 1e-6,
}

In [15]:
# def mean_curvature_loss(model, pts_space):
#     F = model.vf_x(model.params, pts_space).squeeze(1)
#     H = model.vf_xx(model.params, pts_space).squeeze(1)
#     ## Quadratic form
#     FHFT = torch.einsum('bi,bij,bj->b', F, H, F)
#     ## Trace of Hessian
#     trH = torch.einsum('bii->b', H)
#     ## Norm of gradient
#     N = F.square().sum(1).sqrt()
#     ## Mean-curvature
#     mean_curvatures = -(FHFT - N.pow(2)*trH) / (2*N.pow(3))
#     return 0.5*mean_curvatures.square().mean()

### Point-sampler demo

### Define the network and derivatives

Solving this problem requires access to first and second order derivatives wrt. the input $x$ of a neural network $f_\theta(x)$ with parameters $\theta$.

### Main training loop

In [16]:
def sample_true_surface(n_samples):
    # Generate radial and angular coordinates
    r = torch.linspace(0, 1, n_samples, dtype=torch.float64)
    phi = torch.linspace(-torch.pi, torch.pi, n_samples, dtype=torch.float64)
    
    # Create a grid of r and phi
    r, phi = torch.meshgrid(r, phi, indexing='ij')

    # Compute the x, y, z coordinates using the parametric equations
    x, y, z = enneper_surface_polar(r.flatten(), phi.flatten())
    points_on_surface = torch.vstack([x, y, z]).T

    return points_on_surface

def sample_model_surface(implicit_function, bounds, n_samples=128):
    x = torch.linspace(-1, 1, n_samples, dtype=torch.float64)
    y = torch.linspace(-1, 1, n_samples, dtype=torch.float64)
    z = torch.linspace(-1, 1, n_samples, dtype=torch.float64)
    
    xx, yy, zz = torch.meshgrid(x, y, z, indexing='ij')
    grid_points = torch.stack([xx.flatten(), yy.flatten(), zz.flatten()], dim=1)
    
    with torch.no_grad():
        values = implicit_function(grid_points).squeeze()
        mask = torch.isclose(values, torch.zeros_like(values), atol=1e-5)  # Adjust tolerance
        points_on_surface = grid_points[mask]
    
    return points_on_surface

def compute_distance(model, true_implicit, n_steps=10):    

    # Sample points on the true implicit surface
    p_surface_true = sample_true_surface(64)
    p_surface_model = sample_model_surface(model, 128)

    # Compute the integral of squared values over both surfaces
    with torch.no_grad():
        model_surface_values = model(p_surface_true).squeeze().square()
        true_surface_values = true_implicit(p_surface_model).squeeze().square()

    d_model = model_surface_values.mean().sqrt()
    d_true = true_surface_values.mean().sqrt()
    # Combine into the distance metric
    distance = torch.sqrt(d_model**2 + d_true**2).item()

    return distance, p_surface_true, p_surface_model

model = GeneralNet(ks=[3, 32, 32, 1])
model = model.double()
distance, p_surface_true, p_surface_model = compute_distance(model.double(), enneper_level_set)
print(f"Distance d(f_i, f_j): {distance}")


Distance d(f_i, f_j): 2.658499726009629


In [17]:
# Generate the mesh using the level set function
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-2, -2, -2], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 2], dtype=torch.float64),
    chunks=2
)

# Visualization
color = 0xbbbbbb

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=color, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig += k3d.points(p_surface_true.cpu().detach(), point_size=0.05)
fig.display()

Output()

In [18]:
import time

# model = GeneralNet(ks=[3, 32, 32, 1])
model = model.double()
netp = NetWithPartials.create_from_model(model=model, nz=0, nx=3)
params = model.params
loss_over_iters = {}
distance_over_iters = {}
bounds = torch.tensor([[-1,1], [-1,1], [-1,1]], dtype=torch.float64)
# Gauss-Newton:
# loss_weights = {"interface": 1.0, "interface_corners": 10.0, "eikonal": 0.001, "laplacian": 1.0}
optim = HandNGD(model, lr=1e-2, config=config)
# Adam:
# loss_weights = {"interface": 10.0, "interface_corners": 0.0, "eikonal": 0.4, "laplacian": 1.0}
# optim = torch.optim.Adam(model.parameters(), lr=1e-3)
# eikonal weight has to be small compared to h_laplace, see StabEik

for i in (pbar:=trange(5000)):
    optim.zero_grad()

    ## 1) Interface at boundaries
    loss_interface  = 0.5*model.f(params, pts_boundary).square().mean()

    ## 2) Eikonal at boundaries (or everywhere?)
    loss_eikonal  = 0.5*(model.vf_x(params, pts_eikonal).squeeze(1).square().sum(1).sqrt() - 1).square().mean()

    loss_laplacian = 0.5*model.v_f_laplace(params, pts_space).squeeze(1).square().mean()

    if i == 500:
        loss_weights = {"interface": 10.0, "eikonal": 0.001, "laplacian": 1.0, "curvature": 0.1}
        config["loss_weights"] = loss_weights
        optim.config = config

    grid_find_surface, grid_dist_find_surface, init_grid_resolution = precompute_sample_grid(
                10000, bounds, equidistant=True)

    success, (p_surface, y_sel) = find_boundary_points_numerically_with_binsearch(
        netp=netp, 
        z=torch.zeros([1,0], dtype=torch.float64), ## or correct shape with degenerate shape 
        n_steps=10,
        x_grid=grid_find_surface, 
        x_grid_dist=grid_dist_find_surface, 
        level_set=0.0,
        nf_is_density=True,
        resolution=init_grid_resolution
        )
    if success:
        pts_surface = p_surface.data
        pts_surface = torch.cat((pts_boundary, pts_surface))
        config["pts_surface"] = pts_surface
        optim.config = config
    
    loss_curvature = 0.5*model.v_f_mean_curvature(params, pts_surface).squeeze(1).square().mean()
       
    loss = loss_weights["interface"] * loss_interface + loss_weights["eikonal"] * loss_eikonal + loss_weights["laplacian"] * loss_laplacian + loss_weights["curvature"] * loss_curvature
    # loss = loss_weights["interface"] * loss_interface + loss_weights["eikonal"] * loss_eikonal + loss_weights["curvature"] * loss_curvature

    loss.backward()
    with torch.no_grad():
        loss_metric = loss_interface + loss_curvature

        pbar.set_description(f"interface: {loss_interface.item():.2e} "
                            f"eikonal: {loss_eikonal.item():.2e} "
                            f"curvature: {loss_curvature.item():.2e} "
                            f"laplacian: {loss_laplacian.item():.2e} "
                            f"{len(pts_surface)}"
                            )
        loss_over_iters[i] = loss_metric.item()
        # distance_over_iters[i], _, _ = compute_distance(model.double(), enneper_level_set)
        # print(distance_over_iters[i])
        
    # optim.loss = loss_metric
    optim.step()

plt.plot(loss_over_iters.keys(), loss_over_iters.values())
plt.semilogy()
plt.show()

  0%|          | 0/5000 [00:00<?, ?it/s]

/cluster/software/stacks/2024-05/python-cuda/3.11.6/lib/python3.11/site-packages/torch/functional.py:512: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3587.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


KeyboardInterrupt: 

In [19]:
# Generate the mesh using the level set function
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-2, -2, -2], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 2], dtype=torch.float64),
    chunks=2
)

# Visualization
color = 0xbbbbbb

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=color, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig += k3d.points(pts_space.cpu().detach(), color=0x00ff00, point_size=0.05)

fig.display()

Output()

### Visualize the result

In [20]:
def sample_true_surface(n_samples):
    # Generate radial and angular coordinates
    r_min, r_max = -1, 1
    r = torch.linspace(r_min, r_max, n_samples, dtype=torch.float64)
    phi = torch.linspace(-torch.pi, torch.pi, n_samples, dtype=torch.float64)
    
    # Create a grid of r and phi
    r, phi = torch.meshgrid(r, phi, indexing='ij')

    # Compute the x, y, z coordinates using the parametric equations
    x, y, z = enneper_surface_polar(r.flatten(), phi.flatten())
    points_on_surface = torch.vstack([x, y, z]).T

    return points_on_surface

def sample_model_surface(implicit_function, n_samples):
    x = torch.linspace(-1, 1, n_samples, dtype=torch.float64)
    y = torch.linspace(-1, 1, n_samples, dtype=torch.float64)
    z = torch.linspace(-1, 1, n_samples, dtype=torch.float64)
    
    xx, yy, zz = torch.meshgrid(x, y, z, indexing='ij')
    grid_points = torch.stack([xx.flatten(), yy.flatten(), zz.flatten()], dim=1)
    
    with torch.no_grad():
        values = implicit_function(grid_points).squeeze()
        mask = torch.isclose(values, torch.zeros_like(values), atol=1e-5)  # Adjust tolerance
        points_on_surface = grid_points[mask]
    
    return points_on_surface

def compute_distance(model, true_implicit):    

    # Sample points on the true implicit surface
    p_surface_true = sample_true_surface(64)
    p_surface_model = sample_model_surface(model, 64)

    # Compute the integral of squared values over both surfaces
    with torch.no_grad():
        model_surface_values = model(p_surface_true).squeeze().square()
        true_surface_values = true_implicit(p_surface_model).squeeze().square()

    d_model = model_surface_values.mean().sqrt()
    print(d_model)
    d_true = true_surface_values.mean().sqrt()
    print(d_true)

    # Combine into the distance metric
    distance = torch.sqrt(d_model**2 + d_true**2).item()

    return distance, p_surface_true, p_surface_model

distance, p_surface_true, p_surface_model = compute_distance(model.double(), enneper_level_set)
print(f"Distance d(f_i, f_j): {distance}")


tensor(0.0005, device='cuda:0', dtype=torch.float64)
tensor(0.0037, device='cuda:0', dtype=torch.float64)
Distance d(f_i, f_j): 0.0037009695812909873


In [21]:
p_surface_model.shape

torch.Size([9, 3])

In [22]:
# Generate the mesh using the level set function
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-1, -1, -1], dtype=torch.float64),
    bbox_max=torch.tensor([1, 1, 1], dtype=torch.float64),
    chunks=2
)

# Visualization
color = 0xbbbbbb

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=color, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig += k3d.points(p_surface_true.cpu().detach(), point_size=0.05)
fig.display()

Output()

In [23]:
# Generate the mesh using the level set function
verts, faces = get_mesh(
    enneper_level_set, N=128, device=device,
    bbox_min=torch.tensor([-2, -2, -2], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 2], dtype=torch.float64),
    chunks=2
)

# Visualization
color = 0xbbbbbb

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=color, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig += k3d.points(p_surface_model.cpu().detach(), point_size=0.05)
fig.display()

Output()